# W9-D1 概念实验：BlueprintVersion 为什么是制品，而不是配置？

配套阅读：同名 `.md`。这里不复述阅读材料，而是用小规模、可运行的模型检验其中的架构约束。

## 实验问题

**问题 1：同一内容能否得到稳定、可审计的身份？**

模拟 LangChat `blueprint/version.py` 的内容寻址：字段顺序不同不应产生新版本，实际内容改变必须产生新 digest。

In [ ]:
from dataclasses import dataclass, field, replace
from hashlib import sha256
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
rng = np.random.default_rng(202608)
def canonical_digest(payload):
    canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return "sha256:" + sha256(canonical.encode()).hexdigest()
def canonical_digest(payload):
    canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return "sha256:" + sha256(canonical.encode()).hexdigest()

base = {"intent": "查询合同", "effect_policy": "read_only", "contract": "sha256:ac1"}
same_meaning = {"contract": "sha256:ac1", "effect_policy": "read_only", "intent": "查询合同"}
changed = {**base, "effect_policy": "conditional_write"}
for name, item in [("原始", base), ("字段重排", same_meaning), ("策略改变", changed)]:
    print(f"{name:8} -> {canonical_digest(item)[:22]}...")
assert canonical_digest(base) == canonical_digest(same_meaning)
assert canonical_digest(base) != canonical_digest(changed)
print("结论：内容身份与格式噪声脱钩；语义变更留下不同证据。")

## 实验问题

**问题 2：为什么冻结对象比“大家别改”可靠？**

用 frozen dataclass 模拟一个已经入库的 BlueprintVersion，并尝试原地修改。

In [ ]:
@dataclass(frozen=True)
class BlueprintVersion:
    blueprint_id: str
    version: int
    contract_digest: str
    content_digest: str
    originating_candidate_id: str

v1 = BlueprintVersion("lease-query", 1, "sha256:ac1", canonical_digest(base), "candidate-17")
print(v1)
try:
    v1.version = 2
except Exception as exc:
    print(type(exc).__name__ + ": 原地改版本被语言层拒绝")

v2 = BlueprintVersion("lease-query", 2, "sha256:ac1", canonical_digest(changed), "candidate-18")
print("正确变更路径：", v1.version, "->", v2.version, "且保留两个 candidate 谱系")

## 实验问题

**问题 3：不可变版本如何让回溯和回滚变成确定性查询？**

对三次候选变更分别物化版本；统计每个 digest 的可复现身份，而不是读取一个会漂移的“当前配置”。

In [ ]:
candidates = [
    {**base, "intent": "查询合同"},
    {**base, "intent": "查询合同并列出风险"},
    {**base, "effect_policy": "conditional_write"},
]
versions = [BlueprintVersion("lease-query", i + 1, "sha256:ac1", canonical_digest(c), f"candidate-{i+1}") for i, c in enumerate(candidates)]
for version in versions:
    print(f"v{version.version}: {version.content_digest[:18]}... <- {version.originating_candidate_id}")
lookup = {v.content_digest: v for v in versions}
reproduced = lookup[versions[0].content_digest]
assert reproduced.version == 1
print("回到 v1 是精确选择历史 digest，不是猜测当时配置。")

## 实验问题

**问题 4：制品模型在什么地方增加了治理成本，换来了什么？**

模拟 200 次变更。配置模式即时生效；制品模式要求 admission/review，但阻止不合规写策略进入可部署集合。

In [ ]:
effects = rng.choice(["read_only", "conditional_write"], size=200, p=[0.82, 0.18])
configuration_live = len(effects)
artifact_admitted = [e for e in effects if e == "read_only"]
blocked = configuration_live - len(artifact_admitted)
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.bar(["配置：即时生效", "制品：通过门后可用"], [configuration_live, len(artifact_admitted)], color=["#d95f02", "#1b9e77"])
ax.set_ylabel("可进入运行时的变更数")
ax.set_title("Admission 门把不合规变更留在运行时之外")
for i, n in enumerate([configuration_live, len(artifact_admitted)]): ax.text(i, n + 3, str(n), ha="center")
plt.tight_layout(); plt.show()
print(f"被治理门阻止的 conditional_write 变更：{blocked} / {configuration_live}")
print("结论：制品增加流程，但把“可部署”变成可验证的集合。")